# Save Your Work

Before starting, save this notebook to your Google Drive:
1. Click **File** → **Save a copy in Drive**
2. The copy will open automatically
3. Work in the Google Drive copy from now on

---

# Additional Regression Methods

Go beyond linear regression with polynomial and tree-based methods.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_error

## Background: Linear Regression Limitations

In [ ]:
# Load data
housing = fetch_california_housing(as_frame=True)
df = housing.frame

X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Linear baseline
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
lr_r2 = lr.score(X_test_scaled, y_test)

print(f"Linear Regression baseline - Test R²: {lr_r2:.4f}")
print(f"  → Leaves {(1 - lr_r2)*100:.1f}% of variance unexplained")

## Method 1: Polynomial Regression

Adds polynomial terms (x², x³, x·y) to capture non-linear relationships.

Still linear regression, just on transformed features.

In [ ]:
# Polynomial features: degree 2 adds squares and interaction terms
poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_scaled)
X_test_poly = poly.transform(X_test_scaled)

print(f"Original features: {X_train_scaled.shape[1]}")
print(f"Polynomial features: {X_train_poly.shape[1]}")
print(f"  → Added {X_train_poly.shape[1] - X_train_scaled.shape[1]} new features")

# Train on polynomial features
poly_lr = LinearRegression()
poly_lr.fit(X_train_poly, y_train)
poly_r2 = poly_lr.score(X_test_poly, y_test)

print(f"\nPolynomial Regression (degree 2) - Test R²: {poly_r2:.4f}")
print(f"Improvement: {(poly_r2 - lr_r2):.4f}")

## Method 2: Decision Tree Regression

Non-parametric: partitions feature space into rectangular regions.

In [ ]:
# Decision tree: recursively splits features to minimize error
tree = DecisionTreeRegressor(max_depth=5, random_state=42)
tree.fit(X_train_scaled, y_train)
tree_r2 = tree.score(X_test_scaled, y_test)

print(f"Decision Tree (max_depth=5) - Test R²: {tree_r2:.4f}")
print(f"Improvement: {(tree_r2 - lr_r2):.4f}")

## Step 4: Hyperparameter Tuning for Trees

In [ ]:
# max_depth controls complexity
# Too small: underfitting
# Too large: overfitting

depths = range(2, 20)
train_scores = []
test_scores = []

for depth in depths:
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42)
    tree.fit(X_train_scaled, y_train)
    
    train_score = tree.score(X_train_scaled, y_train)
    test_score = tree.score(X_test_scaled, y_test)
    
    train_scores.append(train_score)
    test_scores.append(test_score)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(depths, train_scores, label='Training R²', marker='o')
plt.plot(depths, test_scores, label='Test R²', marker='s')
plt.xlabel('Tree Depth')
plt.ylabel('R²')
plt.title('Decision Tree: Effect of Depth on Performance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Best depth
best_depth = depths[np.argmax(test_scores)]
print(f"Best depth: {best_depth}")
print(f"Test R² at best depth: {max(test_scores):.4f}")

## Step 5: Method Comparison

In [ ]:
print("\n" + "="*60)
print("COMPARISON: Linear vs. Polynomial vs. Decision Tree")
print("="*60)

print(f"\nLinear Regression:")
print(f"  Test R²: {lr_r2:.4f}")
print(f"  Pros: Simple, interpretable")
print(f"  Cons: Assumes linear relationships")

print(f"\nPolynomial Regression (degree 2):")
print(f"  Test R²: {poly_r2:.4f}")
print(f"  Improvement: +{(poly_r2 - lr_r2):.4f}")
print(f"  Pros: Captures non-linearity")
print(f"  Cons: Risk of overfitting, can be slow for high degrees")

best_tree = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
best_tree.fit(X_train_scaled, y_train)
best_tree_r2 = best_tree.score(X_test_scaled, y_test)

print(f"\nDecision Tree (depth {best_depth}):")
print(f"  Test R²: {best_tree_r2:.4f}")
print(f"  Improvement: +{(best_tree_r2 - lr_r2):.4f}")
print(f"  Pros: No scaling needed, handles non-linearity & interactions")
print(f"  Cons: Less interpretable, risk of overfitting")